# 03 — QC and preprocessing

The standard single-cell flow: QC, doublet removal, normalization, then PCA, clustering, and UMAP. The official file is already filtered, but redoing QC shows what was filtered and why.

Reads `data/{TISSUE}/raw.h5ad`, writes `data/{TISSUE}/processed.h5ad`. Findings at the end are Lung's.

### 1. Setup

In [ ]:
import os
import sys
sys.path.append("..")          # so "src" can be imported from the notebook folder

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
from src import aging as ag

TISSUE = os.environ.get("TISSUE", "Lung")   # set by src/run_tissue.py; edit here to run by hand
P = ag.paths(TISSUE)                        # data/{TISSUE}/..., figure/{TISSUE}/...

### 2. Rebuild from raw counts and verify

`.X` is already log-normalized, so start from `.raw` to avoid normalizing twice.

In [ ]:
adata = sc.read_h5ad(P.raw)
counts = adata.raw.to_adata()
counts.obs = adata.obs.copy()
counts.obsm["X_umap_original"] = adata.obsm["X_umap"]   # keep the provided UMAP for comparison
adata = counts

In [ ]:
v = adata.X.data if hasattr(adata.X, "data") else adata.X
print("all integers:", np.allclose(v, np.round(v)), "max:", v.max())   # expect True and a large max

### 3. QC metrics

No mitochondrial genes in the TMS release, so QC uses gene and count thresholds only.

In [ ]:
sc.pp.calculate_qc_metrics(adata, percent_top=None, log1p=False, inplace=True)

In [ ]:
sc.pl.violin(adata, ["n_genes_by_counts", "total_counts"], groupby="age", multi_panel=True, show=False)
plt.savefig(P.figure("qc_by_age.png"), dpi=200, bbox_inches="tight")

In [ ]:
sc.pl.violin(adata, ["n_genes_by_counts"], groupby="mouse.id", rotation=90, show=False)   # QC per mouse too
plt.savefig(P.figure("qc_by_mouse.png"), dpi=200, bbox_inches="tight")

### 4. Filter cells and keep a copy of raw counts

In [ ]:
# The released data is already filtered (minimum ~500 genes per cell), so this is a safety net.
n_before = adata.n_obs
adata = adata[adata.obs.n_genes_by_counts > 300].copy()
print("removed:", n_before - adata.n_obs)

In [ ]:
adata.layers["counts"] = adata.X.copy()   # needed for pseudobulk (Phase 7)

### 5. Doublets

Scrublet normalizes internally, so it runs on raw counts, before Section 6.

In [ ]:
sc.pp.scrublet(adata, batch_key="mouse.id")
print(adata.obs["predicted_doublet"].value_counts())
print(pd.crosstab(adata.obs["mouse.id"], adata.obs["predicted_doublet"]))

In [ ]:
# Mouse IDs like "3-M-5/6" contain "/", which h5ad can't store as a key.
# Scrublet keeps per-batch details in .uns; the per-cell results stay in .obs.
adata.uns.pop("scrublet", None)

adata = adata[~adata.obs["predicted_doublet"]].copy()

### 6. Normalize, log-transform, highly variable genes

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=2000)

### 7. PCA, neighbors, clustering, UMAP

In [ ]:
sc.pp.pca(adata)
sc.pp.neighbors(adata)
sc.tl.leiden(adata, resolution=0.5, flavor="igraph", n_iterations=2, directed=False)
sc.tl.umap(adata)

### 8. Plot

Do Leiden clusters line up with the provided cell types? Do old and young cells mix within each type? If one age or one mouse forms its own island, that may be batch, not biology.

In [ ]:
sc.pl.umap(adata, color=["cell_ontology_class", "leiden", "age", "mouse.id"], ncols=1, show=False)
plt.savefig(P.figure("umap.png"), dpi=200, bbox_inches="tight")

### 9. One mouse at a time

Similar colors are easy to confuse, so plot one mouse in color and the rest in gray. The mice shown are the tissue's reviewed ones (`inspect_mice` in `ag.TISSUE_SETTINGS`, for Lung 30-M-2 and 30-M-5) plus any mouse that supplies over half of a cell type (`ag.flag_outlier_mice`).

In [ ]:
flags = ag.flag_outlier_mice(adata)
print(flags)
inspect = list(dict.fromkeys(ag.settings(TISSUE).get("inspect_mice", []) + list(flags["top_mouse"])))
for m in inspect:
    sc.pl.umap(adata, color="mouse.id", groups=[m], title=m, size=4, show=False)
    plt.savefig(P.figure(f"umap_{ag.slug(m)}.png"), dpi=200, bbox_inches="tight")

### 10. Is the layout difference just UMAP randomness?

If the same groups appear in a rearranged layout, the difference from the provided UMAP is layout, not biology.

In [ ]:
sc.tl.umap(adata, random_state=1, key_added="X_umap_seed1")
sc.pl.embedding(adata, basis="X_umap_seed1", color=["cell_ontology_class", "mouse.id"], ncols=1, show=False)
plt.savefig(P.figure("umap_seed1.png"), dpi=200, bbox_inches="tight")

### 11. Save for the next phases

In [ ]:
adata.write_h5ad(P.processed)

### Findings (Lung)

- **The file is already filtered:** the `> 300` genes filter removed 1 cell. Methods line: "data used as released, pre-filtered by the TMS consortium."
- **Doublets:** Scrublet on raw counts flagged 49 cells (0.2%). 30-M-2 had 0, likely because Scrublet detects mixtures of *different* cell types and that sample is almost one cell type.
- **Final dataset:** 24,490 cells.
- **30-M-2 is almost a myeloid-only sample:** 8,014 cells, about a third of the dataset, nearly all in the monocyte/macrophage island. Its classical monocytes sit apart from the other mice's, which mix well. So the problem is **one mouse, not the cell type**, and a second UMAP seed keeps that separation.
- **30-M-5 is not an outlier:** few cells (504), spread normally across cell types.
- **Fibroblasts and adventitial cells form a mostly-3m island.**